# Refined  Model

## Importing Packages

In [1]:
import re
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import log_loss
%pip install catboost lightgbm
from catboost import CatBoostClassifier, Pool
from lightgbm import LGBMClassifier

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.2 MB/s eta 0:00:00


## Importing data

In [2]:
# Define paths and corresponding raw GitHub URLs
files = {
    'train': {
        'local': '../data/raw/Train.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/Train.csv'
    },
    'test': {
        'local': '../data/raw/Test.csv',
        'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/Test.csv'
    },
    'sample_sub': {
            'local': '../data/raw/SampleSubmission.csv',
            'remote': 'https://raw.githubusercontent.com/MarcelNazare/Tanzania-Tourism-Expenditure-Classsifier/refs/heads/master/data/raw/SampleSubmission.csv'
        }

}

data = {}

# Loop through each dataset to try local loading first, then fallback to GitHub
for name, paths in files.items():
    try:
        data[name] = pd.read_csv(paths['local'])
        print(f"Loaded {name} from local path.")
    except (FileNotFoundError, OSError):
        print(f"Local file for {name} not found. Fetching from GitHub...")
        try:
            data[name] = pd.read_csv(paths['remote'])
            print(f"Loaded {name} from GitHub successfully.")
        except Exception as e:
            if name == 'var_defs':
                print(f"Warning: Could not fetch {name}. Setting to None.")
                data[name] = None
            else:
                raise e

# Unpack the dictionary into individual variables
train = data['train']
test = data['test']
sample_sub = data['sample_sub']

Local file for train not found. Fetching from GitHub...
Loaded train from GitHub successfully.
Local file for test not found. Fetching from GitHub...
Loaded test from GitHub successfully.
Local file for sample_sub not found. Fetching from GitHub...
Loaded sample_sub from GitHub successfully.


## Setting up our columns

In [3]:
id_col = 'Tour_ID'
target_col = 'cost_category'
target_classes = ['High Cost', 'Higher Cost', 'Highest Cost', 'Low Cost', 'Lower Cost', 'Normal Cost']
n_classes = len(target_classes)

class_to_idx = {cls_name: i for i, cls_name in enumerate(target_classes)}
idx_to_class = {i: cls_name for i, cls_name in enumerate(target_classes)}
train['target'] = train[target_col].map(class_to_idx)

## Advanced Feature Engineering Pipeline

In [4]:
def parse_age_ordinal(age_str):
    """Extract the first number in an age-group string (e.g. '25-44' -> 25)
    to give the model an ordinal signal in addition to the raw category."""
    if pd.isna(age_str):
        return np.nan
    match = re.search(r'\d+', str(age_str))
    return float(match.group()) if match else np.nan


def preprocess_dataset(df):
    df = df.copy()

    # Typos & Missing Values
    if 'main_activity' in df.columns:
        df['main_activity'] = df['main_activity'].replace({'Widlife Tourism': 'Wildlife Tourism'})

    df['travel_with'] = df['travel_with'].fillna('Alone')
    df['total_female'] = df['total_female'].fillna(0)
    df['total_male'] = df['total_male'].fillna(0)
    df['most_impressing'] = df.get('most_impressing', pd.Series()).fillna('No Answer')

    # Aggregations & Per-Capita Metrics
    df['total_people'] = df['total_female'] + df['total_male']
    df['total_people_safe'] = df['total_people'].apply(lambda x: 1 if x == 0 else x)

    df['total_nights'] = df['night_mainland'] + df['night_zanzibar']
    df['total_nights_safe'] = df['total_nights'].apply(lambda x: 1 if x == 0 else x)

    # Duration Ratios
    df['mainland_ratio'] = df['night_mainland'] / df['total_nights_safe']
    df['zanzibar_ratio'] = df['night_zanzibar'] / df['total_nights_safe']
    df['female_ratio'] = df['total_female'] / df['total_people_safe']
    df['nights_per_person'] = df['total_nights'] / df['total_people_safe']

    # Package Depth Metrics
    package_cols = [
        'package_transport_int', 'package_accomodation', 'package_food',
        'package_transport_tz', 'package_sightseeing', 'package_guided_tour',
        'package_insurance'
    ]
    df['package_count'] = (df[package_cols] == 'Yes').sum(axis=1)
    df['package_depth'] = df['package_count'] / len(package_cols)
    df['is_full_package'] = (df['package_count'] == len(package_cols)).astype(int)
    df['has_no_package'] = (df['package_count'] == 0).astype(int)

    # --- NEW: scale-compressing transforms (help tree splits on skewed counts) ---
    df['log_total_nights'] = np.log1p(df['total_nights'])
    df['log_total_people'] = np.log1p(df['total_people'])
    df['nights_x_people'] = df['total_nights'] * df['total_people']

    # --- NEW: ordinal parse of age_group, kept alongside the raw category ---
    if 'age_group' in df.columns:
        df['age_group_ordinal'] = df['age_group'].apply(parse_age_ordinal)
        df['age_group_ordinal'] = df['age_group_ordinal'].fillna(df['age_group_ordinal'].median())

    # --- NEW: purpose x tour_arrangement interaction (captures e.g. "Business +
    # Independent" vs "Business + Package" spending differently) ---
    if 'purpose' in df.columns and 'tour_arrangement' in df.columns:
        df['purpose_x_arrangement'] = df['purpose'].astype(str) + '_' + df['tour_arrangement'].astype(str)

    return df


train_df = preprocess_dataset(train)
test_df = preprocess_dataset(test)


## Categorical Feature Specification

In [5]:
cat_features = [
    'country', 'age_group', 'travel_with', 'purpose', 'main_activity',
    'info_source', 'tour_arrangement', 'package_transport_int',
    'package_accomodation', 'package_food', 'package_transport_tz',
    'package_sightseeing', 'package_guided_tour', 'package_insurance', 'first_trip_tz',
    'most_impressing',
    'purpose_x_arrangement',  # new interaction feature, native categorical
]
cat_features = [c for c in cat_features if c in train_df.columns]

for col in cat_features:
    train_df[col] = train_df[col].astype(str)
    test_df[col] = test_df[col].astype(str)

## LEAK-FREE Frequency Encoding: fit on TRAIN ONLY, map to both sets.

In [6]:
for col in ['country', 'purpose', 'main_activity']:
    freq_map = train_df[col].value_counts()
    train_df[f'{col}_freq'] = train_df[col].map(freq_map)
    test_df[f'{col}_freq'] = test_df[col].map(freq_map).fillna(0)  # unseen test categories -> 0

features = [col for col in train_df.columns if col not in [id_col, target_col, 'target']]

X = train_df[features].copy()
y = train_df['target'].values
X_test = test_df[features].copy()

## Stratified 5-Fold Cross-Validation with LightGBM

In [7]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

CATBOOST_SEEDS = [42, 202, 777]   # bag 3 seeds per fold to cut CatBoost variance
te_cols = ['country', 'purpose']
SMOOTHING_K = 15  # shrinkage strength toward the global class prior

oof_preds_cb = np.zeros((len(train_df), n_classes))
oof_preds_lgb = np.zeros((len(train_df), n_classes))
test_preds_cb = np.zeros((len(test_df), n_classes))
test_preds_lgb = np.zeros((len(test_df), n_classes))

global_class_priors = np.bincount(y, minlength=n_classes) / len(y)

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
    print(f"\n--- Training Fold {fold + 1} ---")

    X_tr, y_tr = X.iloc[train_idx].copy(), y[train_idx]
    X_va, y_va = X.iloc[val_idx].copy(), y[val_idx]
    X_te = X_test.copy()

    # --- SMOOTHED in-fold target encoding (Bayesian shrinkage) ---
    for col in te_cols:
        counts = X_tr[col].value_counts()
        for c in range(n_classes):
            class_hits = X_tr[y_tr == c][col].value_counts()
            raw_likelihood = (class_hits / counts).fillna(0)
            shrink = counts / (counts + SMOOTHING_K)
            smoothed = raw_likelihood * shrink + global_class_priors[c] * (1 - shrink)

            X_tr[f'{col}_te_class_{c}'] = X_tr[col].map(smoothed).fillna(global_class_priors[c])
            X_va[f'{col}_te_class_{c}'] = X_va[col].map(smoothed).fillna(global_class_priors[c])
            X_te[f'{col}_te_class_{c}'] = X_te[col].map(smoothed).fillna(global_class_priors[c])

    # ===================== CatBoost: multi-seed bagging =====================
    train_pool = Pool(X_tr, y_tr, cat_features=cat_features)
    val_pool = Pool(X_va, y_va, cat_features=cat_features)
    test_pool = Pool(X_te, cat_features=cat_features)

    fold_val_preds_cb = np.zeros((len(val_idx), n_classes))
    fold_test_preds_cb = np.zeros((len(test_df), n_classes))

    for seed in CATBOOST_SEEDS:
        model = CatBoostClassifier(
            iterations=1800,
            learning_rate=0.03,
            depth=6,
            l2_leaf_reg=5,
            random_strength=0.8,
            bagging_temperature=0.3,
            loss_function='MultiClass',
            eval_metric='MultiClass',
            random_seed=seed,
            verbose=False,
        )
        model.fit(
            train_pool,
            eval_set=val_pool,
            early_stopping_rounds=120,
            use_best_model=True,
        )
        fold_val_preds_cb += model.predict_proba(val_pool) / len(CATBOOST_SEEDS)
        fold_test_preds_cb += model.predict_proba(test_pool) / len(CATBOOST_SEEDS)

    oof_preds_cb[val_idx] = fold_val_preds_cb
    test_preds_cb += fold_test_preds_cb / skf.n_splits

    fold_ll_cb = log_loss(y_va, np.clip(fold_val_preds_cb, 1e-15, 1 - 1e-15), labels=list(range(n_classes)))
    print(f"  CatBoost fold log loss: {fold_ll_cb:.5f}")

    # ===================== LightGBM =====================
    X_tr_lgb, X_va_lgb, X_te_lgb = X_tr.copy(), X_va.copy(), X_te.copy()
    for c in cat_features:
        combined = pd.concat([X_tr_lgb[c], X_va_lgb[c], X_te_lgb[c]]).astype('category')
        cats = combined.cat.categories
        X_tr_lgb[c] = pd.Categorical(X_tr_lgb[c], categories=cats)
        X_va_lgb[c] = pd.Categorical(X_va_lgb[c], categories=cats)
        X_te_lgb[c] = pd.Categorical(X_te_lgb[c], categories=cats)

    lgb_model = LGBMClassifier(
        objective='multiclass',
        num_class=n_classes,
        n_estimators=1800,
        learning_rate=0.03,
        max_depth=6,
        num_leaves=31,
        reg_lambda=5,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42 + fold,
        verbose=-1,
    )
    lgb_model.fit(
        X_tr_lgb, y_tr,
        eval_set=[(X_va_lgb, y_va)],
        eval_metric='multi_logloss',
        categorical_feature=cat_features,
        callbacks=[],
    )

    val_preds_lgb = lgb_model.predict_proba(X_va_lgb)
    oof_preds_lgb[val_idx] = val_preds_lgb
    test_preds_lgb += lgb_model.predict_proba(X_te_lgb) / skf.n_splits

    fold_ll_lgb = log_loss(y_va, np.clip(val_preds_lgb, 1e-15, 1 - 1e-15), labels=list(range(n_classes)))
    print(f"  LightGBM fold log loss: {fold_ll_lgb:.5f}")


--- Training Fold 1 ---
  CatBoost fold log loss: 1.04805


ValueError: pandas dtypes must be int, float or bool.
Fields with bad pandas dtypes: most_impressing: object

## BLEND WEIGHT SEARCH (minimize OOF log loss over CatBoost/LightGBM mix)

In [ ]:
oof_cb_clipped = np.clip(oof_preds_cb, 1e-15, 1 - 1e-15)
oof_cb_clipped = oof_cb_clipped / oof_cb_clipped.sum(axis=1, keepdims=True)

oof_lgb_clipped = np.clip(oof_preds_lgb, 1e-15, 1 - 1e-15)
oof_lgb_clipped = oof_lgb_clipped / oof_lgb_clipped.sum(axis=1, keepdims=True)

best_w, best_ll = 1.0, log_loss(y, oof_cb_clipped)
for w in np.arange(0.0, 1.01, 0.05):
    blend = w * oof_cb_clipped + (1 - w) * oof_lgb_clipped
    ll = log_loss(y, blend)
    if ll < best_ll:
        best_ll, best_w = ll, w

print(f"\nBest blend weight (CatBoost share): {best_w:.2f}  ->  OOF log loss: {best_ll:.5f}")
print(f"  (CatBoost alone: {log_loss(y, oof_cb_clipped):.5f} | LightGBM alone: {log_loss(y, oof_lgb_clipped):.5f})")

## Probability Calibration

In [ ]:
oof_blend = best_w * oof_cb_clipped + (1 - best_w) * oof_lgb_clipped
oof_preds_calibrated = np.clip(oof_blend, 1e-15, 1 - 1e-15)
oof_preds_calibrated = oof_preds_calibrated / oof_preds_calibrated.sum(axis=1, keepdims=True)

test_cb_clipped = np.clip(test_preds_cb, 1e-15, 1 - 1e-15)
test_cb_clipped = test_cb_clipped / test_cb_clipped.sum(axis=1, keepdims=True)
test_lgb_clipped = np.clip(test_preds_lgb, 1e-15, 1 - 1e-15)
test_lgb_clipped = test_lgb_clipped / test_lgb_clipped.sum(axis=1, keepdims=True)

test_blend = best_w * test_cb_clipped + (1 - best_w) * test_lgb_clipped
test_preds_calibrated = np.clip(test_blend, 1e-15, 1 - 1e-15)
test_preds_calibrated = test_preds_calibrated / test_preds_calibrated.sum(axis=1, keepdims=True)

In [ ]:
cv_score = log_loss(y, oof_preds_calibrated)
print(f"\n==========================================")
print(f"FINAL BLENDED OOF LOG LOSS: {cv_score:.5f}")
print(f"==========================================")

## Optimized Model Submission

In [ ]:
submission = pd.DataFrame(test_preds_calibrated, columns=[idx_to_class[i] for i in range(n_classes)])
submission.insert(0, id_col, test_df[id_col])

submission = submission[sample_sub.columns]
submission = sample_sub[[id_col]].merge(submission, on=id_col, how='left')

submission.to_csv('optimized_submission.csv', index=False)
print("Successfully generated file!")